# Hugging Face Transformers Pipelines

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/04-llm-and-transformers/02_huggingface_pipelines.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Use pretrained models for real NLP tasks — text classification, generation, summarization, question answering, and embeddings — without training anything.

**Prerequisites:** Attention and transformers (notebook 01)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q torch transformers


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModel
import torch

## 1. Sentiment Analysis

In [ ]:
classifier = pipeline("sentiment-analysis")

texts = [
    "I love this product, it's absolutely amazing!",
    "This is terrible, worst purchase ever.",
    "It's okay, nothing special but does the job.",
]

results = classifier(texts)
for text, result in zip(texts, results):
    print(f"  '{text[:50]}...' → {result['label']} ({result['score']:.4f})")

## 2. Text Generation

In [ ]:
generator = pipeline("text-generation", model="gpt2")

prompt = "The future of artificial intelligence is"
outputs = generator(prompt, max_new_tokens=50, num_return_sequences=2,
                    temperature=0.8, do_sample=True)

for i, output in enumerate(outputs):
    print(f"Generation {i+1}:")
    print(f"  {output['generated_text']}\n")

## 3. Summarization

In [ ]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

article = """
Machine learning is a subset of artificial intelligence that focuses on building
systems that learn from data. Unlike traditional programming where rules are
explicitly coded, machine learning algorithms identify patterns in data and make
decisions with minimal human intervention. The field has seen tremendous growth
in recent years, driven by advances in computing power, availability of large
datasets, and breakthroughs in deep learning architectures. Applications range
from image recognition and natural language processing to autonomous vehicles
and drug discovery. The transformer architecture, introduced in 2017, has been
particularly impactful, forming the basis of models like GPT and BERT.
"""

summary = summarizer(article, max_length=50, min_length=20)
print("Summary:")
print(f"  {summary[0]['summary_text']}")

## 4. Question Answering

In [ ]:
qa = pipeline("question-answering")

context = """
PyTorch is an open-source machine learning framework developed by Meta AI.
It was released in 2016 and has become one of the most popular frameworks
for deep learning research. PyTorch uses dynamic computational graphs,
which makes it more intuitive for debugging compared to static graph
frameworks. It supports GPU acceleration through CUDA and Apple's MPS backend.
"""

questions = [
    "Who developed PyTorch?",
    "When was PyTorch released?",
    "What type of computational graphs does PyTorch use?",
]

for q in questions:
    answer = qa(question=q, context=context)
    print(f"  Q: {q}")
    print(f"  A: {answer['answer']} (confidence: {answer['score']:.4f})\n")

## 5. Tokenizers Deep Dive

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Transformers are revolutionizing natural language processing!"

encoded = tokenizer(text, return_tensors="pt")
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"Original: {text}")
print(f"Tokens:   {tokens}")
print(f"IDs:      {ids}")
print(f"Decoded:  {tokenizer.decode(encoded['input_ids'][0])}")
print(f"\nSpecial tokens: [CLS]={tokenizer.cls_token_id}, [SEP]={tokenizer.sep_token_id}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

# Compare tokenizers
for model_name in ["bert-base-uncased", "gpt2", "t5-small"]:
    tok = AutoTokenizer.from_pretrained(model_name)
    toks = tok.tokenize(text)
    print(f"\n{model_name}: {len(toks)} tokens → {toks}")

## 6. Working with Embeddings

In [ ]:
model = AutoModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sentences = [
    "The cat sat on the mat",
    "A kitten rested on the rug",
    "The stock market crashed today",
]

def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze()

embeddings = [get_embedding(s) for s in sentences]

# Compute cosine similarity
from torch.nn.functional import cosine_similarity

print("Cosine Similarities:")
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0))
        print(f"  '{sentences[i][:30]}...' vs '{sentences[j][:30]}...' → {sim.item():.4f}")

## Try It Yourself

1. Use the zero-shot classification pipeline to categorize news headlines into topics without any training data.
2. Compare tokenization of the same sentence across 3 different models (BERT, GPT-2, T5). How do they differ?
3. Compute sentence embeddings for 10 sentences and build a simple semantic search — given a query, find the most similar sentence.

In [ ]:
# Your code here